# YOLO26s Detection Training Pipeline

**Pipeline overview:**
1. Reads COCO-format annotations from all dataset sources
2. Converts to YOLO detection format (normalised `.txt` labels)
3. Splits 50 full Self Images into test, rest into train/val
4. Filters bad annotations and handles class mapping
5. Creates `data.yaml` config
6. Trains `yolo26s.pt` detection model
7. Validates on test set with mAP metrics and saves a clean results summary

## 0 · Imports & Dependencies

In [1]:
import subprocess, sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

try:
    import ultralytics
except ImportError:
    print("Installing ultralytics...")
    pip_install("ultralytics")

try:
    import yaml
except ImportError:
    pip_install("pyyaml")

In [2]:
import json
import os
import shutil
import random
import yaml
import datetime
from pathlib import Path
from collections import Counter, defaultdict
from PIL import Image

print("All imports successful")

All imports successful


## 1 · Configuration

Edit the cells below to match your setup before running the rest of the notebook.

In [3]:
# Paths 
REPO_ROOT  = Path(".")          # Root folder containing 'Training Images/'
OUTPUT_DIR = Path("det_dataset") # Where the prepared YOLO dataset will be written

# Class definitions
CLASS_NAMES = ["Bottled_Drink", "Canned", "Fresh_Produce", "Packaged_Food"]
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}

CATEGORY_MAP = {
    1: "Bottled_Drink",
    2: "Canned",
    3: "Fresh_Produce",
    4: "Packaged_Food",
}

# Split settings 
NUM_TEST_SELF = 50   # Number of Self Images held out for the test split
VAL_FRACTION  = 0.15 # Fraction of training pool used for validation
SEED = 42
random.seed(SEED)

In [4]:
# Training hyperparameters 
EPOCHS = 100
BATCH  = 16
IMGSZ  = 640

# Device selection 
DEVICE = "auto"

# Results output 
RUN_NAME    = "yolo26s_det_grocery"  # Sub-folder name inside RESULTS_DIR
RESULTS_DIR = Path("runs/detect")    # Parent folder for all training runs

In [5]:
# Quick sanity-check: show what device will be used
import torch

if DEVICE == "auto":
    if torch.cuda.is_available():
        _dev = f"CUDA ({torch.cuda.get_device_name(0)})"
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        _dev = "MPS (Apple Silicon)"
    else:
        _dev = "CPU"
else:
    _dev = DEVICE

print(f"Device setting : {DEVICE!r}  ->  will run on: {_dev}")
print(f"PyTorch version: {torch.__version__}")

Device setting : 'auto'  ->  will run on: CUDA (NVIDIA GeForce RTX 3060 Laptop GPU)
PyTorch version: 2.11.0+cu130


## 2 · Helper Functions

In [6]:
# Dataset discovery 

DATASET_SPECS = [
    {
        "source": "kaggle",
        "ann": "Training Images/Kaggle Images/annotations.json",
        "img": "Training Images/Kaggle Images/images",
    },
    {
        "source": "mv",
        "ann": "Training Images/MV Images/selected_250.json",
        "img": "Training Images/MV Images/img",
    },
    {
        "source": "self_girvin",
        "ann": "Training Images/Self Images/annotated_resized_Girvin-images/train/_annotations.coco.json",
        "img": "Training Images/Self Images/annotated_resized_Girvin-images/train",
    },
    {
        "source": "self_naveen",
        "ann": "Training Images/Self Images/annotated_resized_Naveen-images/train/_annotations.coco.json",
        "img": "Training Images/Self Images/annotated_resized_Naveen-images/train",
    },
    {
        "source": "self_stanley_add",
        "ann": "Training Images/Self Images/stanley_dataset/stanley_add_dataset/coco_annotations.json",
        "img": "Training Images/Self Images/stanley_dataset/stanley_add_dataset/images",
    },
    {
        "source": "self_stanley_orig",
        "ann": "Training Images/Self Images/stanley_dataset/stanley_original/annotations.json",
        "img": "Training Images/Self Images/stanley_dataset/stanley_original/images",
    },
]


def find_coco_datasets(repo_root: Path):
    """Walk the repo and return list of dataset dicts that actually exist."""
    datasets = []
    for spec in DATASET_SPECS:
        ann_path = repo_root / spec["ann"]
        img_path = repo_root / spec["img"]
        if ann_path.exists():
            datasets.append({"ann_file": ann_path, "img_dir": img_path, "source": spec["source"]})
            print(f"  [✓] {spec['source']}")
        else:
            print(f"  [✗] {spec['source']} — not found at {ann_path}")
    return datasets

In [7]:
# Annotation parsing 

def map_category(cat_id, cat_name):
    """Map a COCO category to our canonical class index (returns None if unknown)."""
    if cat_id in CATEGORY_MAP:
        return CLASS_TO_IDX.get(CATEGORY_MAP[cat_id])
    name_lower = cat_name.lower().replace(" ", "_")
    for canon_name, idx in CLASS_TO_IDX.items():
        if canon_name.lower() in name_lower or name_lower in canon_name.lower():
            return idx
    return None


def parse_coco_dataset(ann_file: Path, img_dir: Path, source: str):
    """
    Parse a COCO JSON and return a list of image records:
    [{ img_path, width, height, labels: [(cls, xc, yc, w, h)], source, img_id }]
    All bbox values are normalised to [0, 1].
    """
    with open(ann_file) as f:
        coco = json.load(f)

    id_to_info = {img["id"]: img for img in coco.get("images", [])}

    cat_id_to_cls = {}
    for cat in coco.get("categories", []):
        cls_idx = map_category(cat["id"], cat["name"])
        if cls_idx is not None:
            cat_id_to_cls[cat["id"]] = cls_idx

    anns_by_img = defaultdict(list)
    skipped = 0

    for ann in coco.get("annotations", []):
        cat_id = ann["category_id"]
        if cat_id not in cat_id_to_cls:
            skipped += 1
            continue

        cls_idx  = cat_id_to_cls[cat_id]
        bbox_raw = ann.get("bbox")

        # Fall back to segmentation polygon if no bbox
        if bbox_raw is None or bbox_raw == []:
            seg = ann.get("segmentation", [])
            if seg and isinstance(seg, list) and len(seg) > 0:
                pts = seg[0] if isinstance(seg[0], list) else seg
                try:
                    xs = [float(pts[i]) for i in range(0, len(pts), 2)]
                    ys = [float(pts[i]) for i in range(1, len(pts), 2)]
                    bbox_raw = [min(xs), min(ys), max(xs) - min(xs), max(ys) - min(ys)]
                except (ValueError, IndexError):
                    skipped += 1
                    continue
            else:
                skipped += 1
                continue

        try:
            if isinstance(bbox_raw, list) and len(bbox_raw) == 1 and isinstance(bbox_raw[0], list):
                bbox_raw = bbox_raw[0]
            x, y, w, h = [float(v) for v in bbox_raw]
        except (ValueError, TypeError):
            skipped += 1
            continue

        if w <= 0 or h <= 0:
            skipped += 1
            continue

        anns_by_img[ann["image_id"]].append((cls_idx, x, y, w, h))

    records = []
    missing = 0

    for img_id, labels in anns_by_img.items():
        info = id_to_info.get(img_id)
        if info is None:
            missing += 1
            continue

        img_path = img_dir / info["file_name"]
        if not img_path.exists():
            alt = img_dir.parent / info["file_name"]
            if alt.exists():
                img_path = alt
            else:
                missing += 1
                continue

        img_w = info.get("width")
        img_h = info.get("height")
        if not img_w or not img_h:
            try:
                with Image.open(img_path) as im:
                    img_w, img_h = im.size
            except Exception:
                missing += 1
                continue

        yolo_labels = []
        for cls_idx, bx, by, bw, bh in labels:
            xc = max(0.0, min(1.0, (bx + bw / 2) / img_w))
            yc = max(0.0, min(1.0, (by + bh / 2) / img_h))
            nw = max(0.001, min(1.0, bw / img_w))
            nh = max(0.001, min(1.0, bh / img_h))
            if nw < 0.005 and nh < 0.005:  # skip sub-pixel boxes
                continue
            yolo_labels.append((cls_idx, xc, yc, nw, nh))

        if yolo_labels:
            records.append({
                "img_path": img_path, "width": img_w, "height": img_h,
                "labels": yolo_labels, "source": source, "img_id": img_id,
            })

    if skipped:
        print(f"    {skipped} annotations skipped (bad bbox / unknown category)")
    if missing:
        print(f"    {missing} images not found on disk")

    return records

In [8]:
# Dataset writing 

def write_yolo_split(records, split_name, output_dir):
    img_out = output_dir / "images" / split_name
    lbl_out = output_dir / "labels" / split_name
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for i, rec in enumerate(records):
        suffix   = rec["img_path"].suffix
        new_name = f"{rec['source']}_{rec['img_id']}_{i:05d}{suffix}"
        shutil.copy2(rec["img_path"], img_out / new_name)
        lbl_path = lbl_out / (Path(new_name).stem + ".txt")
        with open(lbl_path, "w") as f:
            for cls_idx, xc, yc, w, h in rec["labels"]:
                f.write(f"{cls_idx} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")

    return len(records)


def write_data_yaml(output_dir: Path):
    data = {
        "path":  str(output_dir.resolve()),
        "train": "images/train",
        "val":   "images/val",
        "test":  "images/test",
        "nc":    len(CLASS_NAMES),
        "names": CLASS_NAMES,
    }
    yaml_path = output_dir / "data.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(data, f, default_flow_style=False)
    return yaml_path

## 3 · Prepare Dataset

In [9]:
print("Discovering datasets...")
datasets = find_coco_datasets(REPO_ROOT)

if not datasets:
    raise RuntimeError(" No datasets found. Check REPO_ROOT.")

print(f"\n{len(datasets)} source(s) found.")

Discovering datasets...
  [✓] kaggle
  [✓] mv
  [✓] self_girvin
  [✓] self_naveen
  [✓] self_stanley_add
  [✓] self_stanley_orig

6 source(s) found.


In [10]:
print("Parsing COCO annotations -> YOLO format...\n")

all_records  = []
self_records = []

for ds in datasets:
    print(f"  {ds['source']}")
    records = parse_coco_dataset(ds["ann_file"], ds["img_dir"], ds["source"])
    n_boxes = sum(len(r["labels"]) for r in records)
    print(f"    -> {len(records)} images, {n_boxes} boxes")
    all_records.extend(records)
    if ds["source"].startswith("self_"):
        self_records.extend(records)

total_boxes = sum(len(r["labels"]) for r in all_records)
print(f"\nTotal  : {len(all_records)} images, {total_boxes} boxes")
print(f"Self   : {len(self_records)} images")

class_counts = Counter()
for r in all_records:
    for cls_idx, *_ in r["labels"]:
        class_counts[CLASS_NAMES[cls_idx]] += 1

print("\nOverall class distribution:")
for cls in CLASS_NAMES:
    pct = 100 * class_counts[cls] / total_boxes if total_boxes else 0
    print(f"  {cls:<20} {class_counts[cls]:>6}  ({pct:.1f}%)")

Parsing COCO annotations -> YOLO format...

  kaggle
    -> 200 images, 2078 boxes
  mv
    -> 250 images, 1290 boxes
  self_girvin
    -> 21 images, 53 boxes
  self_naveen
    -> 35 images, 85 boxes
  self_stanley_add
    -> 9 images, 23 boxes
  self_stanley_orig
    -> 32 images, 62 boxes

Total  : 547 images, 3591 boxes
Self   : 97 images

Overall class distribution:
  Bottled_Drink          1418  (39.5%)
  Canned                 1421  (39.6%)
  Fresh_Produce           526  (14.6%)
  Packaged_Food           226  (6.3%)


In [11]:
print("Splitting dataset...\n")

random.shuffle(self_records)
n_test       = min(NUM_TEST_SELF, len(self_records))
test_records = self_records[:n_test]
remaining    = self_records[n_test:]

non_self   = [r for r in all_records if not r["source"].startswith("self_")]
train_pool = non_self + remaining
random.shuffle(train_pool)

val_size     = max(1, int(len(train_pool) * VAL_FRACTION))
val_records  = train_pool[:val_size]
train_records = train_pool[val_size:]

splits = {"train": train_records, "val": val_records, "test": test_records}

print(f"  {'Split':<8}  {'Images':>7}  {'Boxes':>7}")
print(f"  {'─'*8}  {'─'*7}  {'─'*7}")
for split, recs in splits.items():
    boxes = sum(len(r["labels"]) for r in recs)
    src   = "(Self only)" if split == "test" else ""
    print(f"  {split:<8}  {len(recs):>7}  {boxes:>7}  {src}")

Splitting dataset...

  Split      Images    Boxes
  ────────  ───────  ───────
  train         423     3000  
  val            74      468  
  test           50      123  (Self only)


In [12]:
print("Writing YOLO dataset to disk...")

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

for split_name, recs in splits.items():
    n = write_yolo_split(recs, split_name, OUTPUT_DIR)
    print(f"  {split_name}: {n} images written")

yaml_path = write_data_yaml(OUTPUT_DIR)
print(f"\n Dataset ready at: {OUTPUT_DIR.resolve()}")
print(f"   data.yaml      : {yaml_path}")

Writing YOLO dataset to disk...
  train: 423 images written
  val: 74 images written
  test: 50 images written

 Dataset ready at: /var/home/polygon/EE3703_RPC/det_dataset
   data.yaml      : det_dataset/data.yaml


In [13]:
# Per-split class distribution table
print(f"\n{'Class':<22}", end="")
for split in splits:
    print(f"  {split:>10}", end="")
print()
print("─" * (22 + 12 * len(splits)))

for cls in CLASS_NAMES:
    print(f"{cls:<22}", end="")
    for recs in splits.values():
        cnt = sum(1 for r in recs for ci, *_ in r["labels"] if CLASS_NAMES[ci] == cls)
        print(f"  {cnt:>10}", end="")
    print()


Class                        train         val        test
──────────────────────────────────────────────────────────
Bottled_Drink                 1218         173          27
Canned                        1229         169          23
Fresh_Produce                  377          95          54
Packaged_Food                  176          31          19


## 4 · Train Model

In [14]:
from ultralytics import YOLO

model = YOLO("yolo26s.pt")
print(f"Loaded model: yolo26s.pt")
print(f"Device      : {DEVICE!r}")
print(f"Epochs      : {EPOCHS}")
print(f"Batch       : {BATCH}")
print(f"Image size  : {IMGSZ}")

Loaded model: yolo26s.pt
Device      : 'auto'
Epochs      : 100
Batch       : 16
Image size  : 640


In [15]:
results = model.train(
    data    = str(yaml_path.resolve()),
    epochs  = EPOCHS,
    imgsz   = IMGSZ,
    batch   = BATCH,
    device  = DEVICE,
    # Early stopping
    patience = 20,
    # Optimiser
    optimizer      = "MuSGD",
    lr0            = 0.01,
    lrf            = 0.001,
    weight_decay   = 0.0005,
    warmup_epochs  = 5,
    warmup_momentum = 0.8,
    # Augmentation
    hsv_h      = 0.015,
    hsv_s      = 0.7,
    hsv_v      = 0.4,
    degrees    = 10.0,
    translate  = 0.1,
    scale      = 0.5,
    fliplr     = 0.5,
    flipud     = 0.0,
    mosaic     = 1.0,
    mixup      = 0.1,
    copy_paste = 0.1,
    erasing    = 0.1,
    # Output location
    name     = RUN_NAME,
    exist_ok = True,
    verbose  = True,
)

save_dir = Path(results.save_dir)
best_pt  = save_dir / "weights" / "best.pt"
print(f"\n Training complete!")
print(f"   Run directory : {save_dir}")
print(f"   Best weights  : {best_pt}")

New https://pypi.org/project/ultralytics/8.4.33 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.31 🚀 Python-3.11.15 torch-2.11.0+cu130 CUDA:auto (NVIDIA GeForce RTX 3060 Laptop GPU, 5781MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/var/home/polygon/EE3703_RPC/det_dataset/data.yaml, degrees=10.0, deterministic=True, device=auto, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.1, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo26s.pt, momentum=0.9

## 5 · Evaluate on Test Set

In [37]:
best_model   = YOLO(str(best_pt))
test_metrics = best_model.val(
    data    = str(yaml_path.resolve()),
    split   = "test",
    imgsz   = IMGSZ,
    device  = DEVICE,
    name     = f"{RUN_NAME}/test_eval",
    exist_ok = True,
    iou=0.5
)

print("Evaluation complete")

Ultralytics 8.4.31 🚀 Python-3.11.15 torch-2.11.0+cu130 CUDA:auto (NVIDIA GeForce RTX 3060 Laptop GPU, 5781MiB)
YOLO26s summary (fused): 122 layers, 9,466,728 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5191.8±1004.3 MB/s, size: 128.5 KB)
val: Scanning /var/home/polygon/EE3703_RPC/det_dataset/labels/test.cache... 50 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 50/50 21.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 4.5it/s 0.9s0.4s
                   all         50        123      0.946      0.919      0.966       0.85
         Bottled_Drink         25         27      0.913      0.963      0.973      0.876
                Canned         21         23      0.976      0.913      0.968      0.824
         Fresh_Produce         27         54      0.958      0.852      0.941      0.823
         Packaged_Food         17         19      0.936      0.947      0.982     

In [38]:
# Pretty-print results 
box = test_metrics.box

header = "Test Set Results"
print(f"\n{'─'*40}")
print(f"  {header}")
print(f"{'─'*40}")
print(f"  {'mAP@0.50':<18}  {box.map50:.4f}")
print(f"  {'mAP@0.50:0.95':<18}  {box.map:.4f}")
print(f"  {'Precision':<18}  {box.mp:.4f}")
print(f"  {'Recall':<18}  {box.mr:.4f}")
print(f"{'─'*40}")
print(f"  Per-class mAP@0.50")
for i, cls_name in enumerate(CLASS_NAMES):
    if i < len(box.ap50):
        print(f"    {cls_name:<20}  {box.ap50[i]:.4f}")
print(f"{'─'*40}")


────────────────────────────────────────
  Test Set Results
────────────────────────────────────────
  mAP@0.50            0.9661
  mAP@0.50:0.95       0.8505
  Precision           0.9461
  Recall              0.9190
────────────────────────────────────────
  Per-class mAP@0.50
    Bottled_Drink         0.9734
    Canned                0.9676
    Fresh_Produce         0.9415
    Packaged_Food         0.9818
────────────────────────────────────────


## 5.1 · Failure Image Comparison

For every test image the model's predictions are matched against ground-truth
boxes (greedy IoU matching, threshold = 0.50). Each box is categorised:

| Colour | Meaning |
|--------|---------|
| 🟩 Green  | Ground-truth box |
| 🟦 Blue   | True Positive – correct class |
| 🟠 Orange | False Positive – no matching GT |
| 🔴 Red    | False Negative – GT box missed |
| 🟣 Purple | IoU match but wrong class |

Images with at least one failure are saved side-by-side (GT | Predictions)
to `<run_dir>/failure_analysis/`, ranked by number of failures.

In [44]:
# Failure analysis helpers 
from PIL import Image, ImageDraw

IOU_THRESH   = 0.50   # IoU required to call a box a match
CONF_THRESH  = 0.25   # Minimum prediction confidence to consider
MAX_FAILURES = 30     # Max number of failure images to save
PANEL_W      = 640    # Width (px) of each panel in the side-by-side image

FAIL_COLOURS = {
    "gt":        "#00CC44",   # green  – ground truth
    "tp":        "#3399FF",   # blue   – true positive
    "fp":        "#FF6600",   # orange – false positive
    "fn":        "#FF2222",   # red    – false negative / missed GT
    "wrong_cls": "#CC00CC",   # purple – right location, wrong class
}


def yolo_to_xyxy(xc, yc, w, h, img_w, img_h):
    """Normalised YOLO xywh  →  pixel xyxy."""
    return (
        (xc - w / 2) * img_w, (yc - h / 2) * img_h,
        (xc + w / 2) * img_w, (yc + h / 2) * img_h,
    )


def box_iou(a, b):
    """IoU between two (x1,y1,x2,y2) tuples."""
    ix1 = max(a[0], b[0]); iy1 = max(a[1], b[1])
    ix2 = min(a[2], b[2]); iy2 = min(a[3], b[3])
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    if inter == 0:
        return 0.0
    return inter / (
        (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    )


def load_gt_boxes(label_path, img_w, img_h):
    """Read a YOLO label file and return [(cls, x1,y1,x2,y2), ...]."""
    boxes = []
    p = Path(label_path)
    if not p.exists():
        return boxes
    with open(p) as fh:
        for line in fh:
            parts = line.strip().split()
            if len(parts) == 5:
                c, xc, yc, bw, bh = int(parts[0]), *map(float, parts[1:])
                boxes.append((c, *yolo_to_xyxy(xc, yc, bw, bh, img_w, img_h)))
    return boxes


def draw_box_pil(draw, xyxy, colour, label, lw=3, dash=None, fill_alpha=40):
    """
    Draw a bounding box with optional dashed outline and semi-transparent fill.
    dash: None = solid, 'dash' = dashed, 'dot' = dotted
    fill_alpha: 0-255 transparency of the interior tint
    """
    x1, y1, x2, y2 = (int(v) for v in xyxy)

    # ── semi-transparent fill ─────────────────────────────────────────────────
    r, g, b = tuple(int(colour.lstrip("#")[i:i+2], 16) for i in (0, 2, 4))
    overlay = Image.new("RGBA", draw.im.size, (0, 0, 0, 0))
    ov_draw = ImageDraw.Draw(overlay)
    ov_draw.rectangle([x1, y1, x2, y2], fill=(r, g, b, fill_alpha))
    # We need the base image to composite onto — store reference via a closure;
    # caller passes it in as draw._img (set in make_panel below)
    if hasattr(draw, "_img"):
        draw._img.paste(
            Image.alpha_composite(draw._img.convert("RGBA"), overlay).convert("RGB"),
            (0, 0)
        )

    # ── outline ───────────────────────────────────────────────────────────────
    if dash is None:
        for d in range(lw):
            draw.rectangle([x1-d, y1-d, x2+d, y2+d], outline=colour)

    elif dash == "dash":
        seg, gap = 12, 6
        for d in range(lw):
            for side_pts in [
                [(x1-d + i, y1-d, min(x1-d+i+seg, x2+d), y1-d) for i in range(0, x2-x1+2*d, seg+gap)],
                [(x1-d + i, y2+d, min(x1-d+i+seg, x2+d), y2+d) for i in range(0, x2-x1+2*d, seg+gap)],
                [(x1-d, y1-d + i, x1-d, min(y1-d+i+seg, y2+d)) for i in range(0, y2-y1+2*d, seg+gap)],
                [(x2+d, y1-d + i, x2+d, min(y1-d+i+seg, y2+d)) for i in range(0, y2-y1+2*d, seg+gap)],
            ]:
                for seg_coords in side_pts:
                    draw.line(seg_coords, fill=colour, width=lw)

    elif dash == "dot":
        dot, gap = 4, 6
        for d in range(lw):
            for side_pts in [
                [(x1-d + i, y1-d) for i in range(0, x2-x1+2*d, dot+gap)],
                [(x1-d + i, y2+d) for i in range(0, x2-x1+2*d, dot+gap)],
                [(x1-d,     y1-d + i) for i in range(0, y2-y1+2*d, dot+gap)],
                [(x2+d,     y1-d + i) for i in range(0, y2-y1+2*d, dot+gap)],
            ]:
                for px, py in side_pts:
                    draw.ellipse([px-1, py-1, px+1, py+1], fill=colour)

    # ── label chip ───────────────────────────────────────────────────────────
    pad, th = 3, 16
    tw = len(label) * 7 + pad * 2
    tx = min(x1, draw.im.size[0] - tw - 4)
    ty = max(0, y1 - th - pad)
    draw.rectangle([tx, ty, tx + tw, ty + th], fill=colour)
    draw.text((tx + pad, ty + pad), label, fill="white")


def make_panel(img_path, annotated_boxes, title, W=PANEL_W):
    """
    annotated_boxes: [(colour, xyxy, label, dash_style), ...]
    dash_style: None | 'dash' | 'dot'
    """
    img = Image.open(img_path).convert("RGB")
    orig_w, orig_h = img.size
    scale = W / orig_w
    img = img.resize((W, int(orig_h * scale)), Image.BILINEAR)
    draw = ImageDraw.Draw(img)
    draw._img = img   # give draw_box_pil access for fill compositing

    for colour, xyxy, label, dash_style in annotated_boxes:
        sx = [xyxy[0]*scale, xyxy[1]*scale, xyxy[2]*scale, xyxy[3]*scale]
        draw_box_pil(draw, sx, colour, label, dash=dash_style)

    bar = Image.new("RGB", (W, 28), "#1a1a2e")
    ImageDraw.Draw(bar).text((6, 7), title, fill="#ffffff")
    return Image.fromarray(np.vstack([np.array(bar), np.array(img)]))

def make_legend(width):
    """Return a narrow legend strip."""
    items = [
        ("GT (ground-truth)",        FAIL_COLOURS["gt"]),
        ("TP (correct class)",        FAIL_COLOURS["tp"]),
        ("FP (no GT match)",          FAIL_COLOURS["fp"]),
        ("FN (missed GT)",            FAIL_COLOURS["fn"]),
        ("Wrong class (IoU ok)",      FAIL_COLOURS["wrong_cls"]),
    ]
    H = 28
    leg = Image.new("RGB", (width, H), "#111111")
    d = ImageDraw.Draw(leg)
    x = 10
    for label, colour in items:
        d.rectangle([x, 7, x+16, 21], fill=colour, outline="white")
        d.text((x+20, 8), label, fill="white")
        x += len(label)*7 + 48
    return leg


print("Failure analysis helpers defined.")

Failure analysis helpers defined.


In [49]:
import numpy as np

# Run failure detection over the entire test set 

test_img_dir = OUTPUT_DIR / "images" / "test"
test_lbl_dir = OUTPUT_DIR / "labels" / "test"
fail_dir     = save_dir / "failure_analysis"
fail_dir.mkdir(exist_ok=True)

test_images = sorted(test_img_dir.glob("*.*"))
print(f"Scanning {len(test_images)} test images for failures…\n")

failure_records = []   # (n_failures, img_name, side_by_side_PIL)
error_counts    = Counter()  # aggregate FP / FN / wrong_cls tallies

for img_path in test_images:
    lbl_path = test_lbl_dir / (img_path.stem + ".txt")

    # ground truth 
    with Image.open(img_path) as im:
        img_w, img_h = im.size
    gt_boxes = load_gt_boxes(lbl_path, img_w, img_h)   # [(cls, x1,y1,x2,y2)]

    # predictions 
    result = best_model.predict(
        str(img_path), imgsz=IMGSZ, conf=CONF_THRESH,
        device=DEVICE, verbose=False
    )[0]
    pred_boxes = []   # [(cls, conf, x1,y1,x2,y2)]
    if result.boxes is not None and len(result.boxes):
        for b in result.boxes:
            cls  = int(b.cls[0].item())
            conf = float(b.conf[0].item())
            x1, y1, x2, y2 = b.xyxy[0].tolist()
            pred_boxes.append((cls, conf, x1, y1, x2, y2))

    # greedy IoU matching 
    iou_matrix = np.zeros((len(pred_boxes), len(gt_boxes)))
    for pi, (pc, pconf, *pb) in enumerate(pred_boxes):
        for gi, (gc, *gb) in enumerate(gt_boxes):
            iou_matrix[pi, gi] = box_iou(pb, gb)

    gt_matched   = [False] * len(gt_boxes)
    pred_matched = [False] * len(pred_boxes)
    matched_pairs = []   # (pi, gi)

    pairs = sorted(
        [(iou_matrix[pi, gi], pi, gi)
         for pi in range(len(pred_boxes))
         for gi in range(len(gt_boxes))],
        reverse=True
    )
    for iou_val, pi, gi in pairs:
        if iou_val < IOU_THRESH:
            break
        if pred_matched[pi] or gt_matched[gi]:
            continue
        pred_matched[pi] = True
        gt_matched[gi]   = True
        matched_pairs.append((pi, gi))

    # build drawing lists 
    gt_draws   = []
    pred_draws = []
    n_fail = 0

    # GT panel – always show all GT boxes in green (solid)
    for gc, *gbox in gt_boxes:
        gt_draws.append((FAIL_COLOURS["gt"], gbox, CLASS_NAMES[gc], None))

    matched_pi_to_gi = {pi: gi for pi, gi in matched_pairs}
    matched_pi_set   = set(matched_pi_to_gi)

    # Prediction panel
    for pi, (pc, pconf, *pb) in enumerate(pred_boxes):
        label = f"{CLASS_NAMES[pc]} {pconf:.2f}"
        if pi in matched_pi_set:
            gi = matched_pi_to_gi[pi]
            gc = gt_boxes[gi][0]
            if pc == gc:
                pred_draws.append((FAIL_COLOURS["tp"], pb, label, "dash"))
            else:
                pred_draws.append(
                    (FAIL_COLOURS["wrong_cls"], pb,
                     f"WRONG:{CLASS_NAMES[pc]}>{CLASS_NAMES[gc]} {pconf:.2f}", "dash")
                )
                error_counts["wrong_cls"] += 1
                n_fail += 1
        else:
            pred_draws.append((FAIL_COLOURS["fp"], pb, f"FP:{label}", "dash"))
            error_counts["fp"] += 1
            n_fail += 1

    # FN: unmatched GT boxes → drawn in red on the prediction panel (dotted)
    for gi, (gc, *gb) in enumerate(gt_boxes):
        if not gt_matched[gi]:
            pred_draws.append((FAIL_COLOURS["fn"], gb, f"MISS:{CLASS_NAMES[gc]}", "dot"))
            error_counts["fn"] += 1
            n_fail += 1

    if n_fail == 0:
        continue

    # compose side-by-side image 
    fn_c  = sum(1 for c,*_ in pred_draws if c == FAIL_COLOURS["fn"])
    fp_c  = sum(1 for c,*_ in pred_draws if c == FAIL_COLOURS["fp"])
    wc_c  = sum(1 for c,*_ in pred_draws if c == FAIL_COLOURS["wrong_cls"])

    gt_panel   = make_panel(img_path, gt_draws,
                            "Ground Truth")
    pred_panel = make_panel(img_path, pred_draws,
                            f"Predictions  ·  FP={fp_c}  FN={fn_c}  WrongClass={wc_c}")

    # pad to equal height before hstack
    h = max(gt_panel.height, pred_panel.height)
    def _pad(im, target_h):
        if im.height >= target_h:
            return im
        pad = Image.new("RGB", (im.width, target_h - im.height), "#000000")
        return Image.fromarray(np.vstack([np.array(im), np.array(pad)]))

    combined = Image.fromarray(
        np.hstack([np.array(_pad(gt_panel, h)), np.array(_pad(pred_panel, h))])
    )
    failure_records.append((n_fail, img_path.name, combined))

# sort worst-first
failure_records.sort(key=lambda x: -x[0])

print(f"Failures found in {len(failure_records)} / {len(test_images)} images")
print(f"  False Positives  : {error_counts['fp']}")
print(f"  False Negatives  : {error_counts['fn']}")
print(f"  Wrong Class      : {error_counts['wrong_cls']}")

Scanning 50 test images for failures…

Failures found in 7 / 50 images
  False Positives  : 2
  False Negatives  : 6
  Wrong Class      : 1


In [50]:
# Save failure comparison images to disk 

legend = make_legend(PANEL_W * 2)

for rank, (n_fail, img_name, combined) in enumerate(failure_records[:MAX_FAILURES]):
    leg = legend.resize((combined.width, legend.height))
    final = Image.fromarray(
        np.vstack([np.array(combined), np.array(leg)])
    )
    out_name = f"rank{rank+1:02d}_nerr{n_fail}_{img_name}"
    final.save(fail_dir / out_name)

n_saved = min(len(failure_records), MAX_FAILURES)
print(f"Saved {n_saved} failure images -> {fail_dir}")
print()
print(f"  {'Rank':<5}  {'Errors':>6}  Image")
print(f"  {'─'*5}  {'─'*6}  {'─'*45}")
for rank, (n_fail, img_name, _) in enumerate(failure_records[:MAX_FAILURES]):
    print(f"  {rank+1:<5}  {n_fail:>6}  {img_name}")

Saved 7 failure images -> /var/home/polygon/EE3703_RPC/runs/detect/yolo26s_det_grocery/failure_analysis

  Rank   Errors  Image
  ─────  ──────  ─────────────────────────────────────────────
  1           3  self_naveen_33_00016.JPG
  2           1  self_naveen_13_00004.JPG
  3           1  self_naveen_39_00020.JPG
  4           1  self_naveen_6_00005.JPG
  5           1  self_stanley_add_4_00001.jpeg
  6           1  self_stanley_add_7_00002.jpeg
  7           1  self_stanley_add_8_00010.jpeg


In [51]:
# Display top-N failure comparisons inline 
import matplotlib.pyplot as plt

N_SHOW = min(6, len(failure_records))

if N_SHOW == 0:
    print("No failures found – the model is perfect on this test set!")
else:
    fig, axes = plt.subplots(N_SHOW, 1, figsize=(18, 5 * N_SHOW),
                             constrained_layout=True)
    if N_SHOW == 1:
        axes = [axes]

    for ax, (n_fail, img_name, combined) in zip(axes, failure_records[:N_SHOW]):
        ax.imshow(np.array(combined))
        ax.set_title(f"[{n_fail} error(s)]  {img_name}", fontsize=11, pad=4)
        ax.axis("off")

    fig.suptitle(
        f"Failure Image Comparison  --  showing {N_SHOW} worst of {len(failure_records)} failure images\n"
        "[Green=GT]  [Blue=TP]  [Orange=FP]  [Red=FN missed]  [Purple=Wrong class]",
        fontsize=12, y=1.01
    )

    overview_path = fail_dir / "failure_overview.png"
    fig.savefig(overview_path, dpi=90, bbox_inches="tight")
    plt.show()
    print(f"Overview grid saved -> {overview_path}")

<Figure size 1800x3000 with 6 Axes>

Overview grid saved -> /var/home/polygon/EE3703_RPC/runs/detect/yolo26s_det_grocery/failure_analysis/failure_overview.png


## 6 · Save Results Summary

In [54]:
import csv

# Gather everything into one dict 
summary = {
    # Run metadata
    "run_name":       RUN_NAME,
    "timestamp":      datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "best_weights":   str(best_pt.resolve()),
    "device":         DEVICE,
    # Hyperparameters
    "epochs":         EPOCHS,
    "batch":          BATCH,
    "imgsz":          IMGSZ,
    # Dataset sizes
    "train_images":   len(train_records),
    "val_images":     len(val_records),
    "test_images":    len(test_records),
    "train_boxes":    sum(len(r["labels"]) for r in train_records),
    "val_boxes":      sum(len(r["labels"]) for r in val_records),
    "test_boxes":     sum(len(r["labels"]) for r in test_records),
    # Overall metrics
    "map50":          round(float(box.map50), 4),
    "map50_95":       round(float(box.map),   4),
    "precision":      round(float(box.mp),    4),
    "recall":         round(float(box.mr),    4),
}

# Per-class AP
per_class_ap = {}
for i, cls_name in enumerate(CLASS_NAMES):
    if i < len(box.ap50):
        key = f"ap50_{cls_name}"
        val = round(float(box.ap50[i]), 4)
        summary[key]       = val
        per_class_ap[cls_name] = val

# JSON summary (human-readable, structured)
json_path = save_dir / "results_summary.json"
with open(json_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"JSON summary saved -> {json_path}")

# CSV row (easy to append across multiple runs)
csv_path = RESULTS_DIR / "all_runs.csv"
write_header = not csv_path.exists()
with open(csv_path, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(summary.keys()))
    if write_header:
        writer.writeheader()
    writer.writerow(summary)
print(f"CSV row appended  -> {csv_path}")

# Markdown report 
md_lines = [
    f"# Results: {RUN_NAME}",
    f"_Generated: {summary['timestamp']}_",
    "",
    "## Run Configuration",
    f"| Parameter | Value |",
    f"|-----------|-------|" ,
    f"| Model     | yolo26s.pt |",
    f"| Device    | {DEVICE} |",
    f"| Epochs    | {EPOCHS} |",
    f"| Batch     | {BATCH} |",
    f"| Image size | {IMGSZ} |",
    f"| Best weights | `{best_pt}` |",
    "",
    "## Dataset Split",
    "| Split | Images | Boxes |",
    "|-------|--------|-------|" ,
    f"| Train | {summary['train_images']} | {summary['train_boxes']} |",
    f"| Val   | {summary['val_images']}   | {summary['val_boxes']}   |",
    f"| Test  | {summary['test_images']}  | {summary['test_boxes']}  |",
    "",
    "## Test Set Metrics",
    "| Metric | Value |",
    "|--------|-------|" ,
    f"| mAP@0.50     | {summary['map50']} |",
    f"| mAP@0.50:0.95 | {summary['map50_95']} |",
    f"| Precision    | {summary['precision']} |",
    f"| Recall       | {summary['recall']} |",
    "",
    "## Per-Class AP@0.50",
    "| Class | AP@0.50 |",
    "|-------|---------|" ,
]
for cls_name, ap in per_class_ap.items():
    md_lines.append(f"| {cls_name} | {ap} |")

md_path = save_dir / "results_summary.md"
with open(md_path, "w") as f:
    f.write("\n".join(md_lines) + "\n")
print(f"Markdown report   -> {md_path}")

print("\n All result files saved.")

JSON summary saved -> /var/home/polygon/EE3703_RPC/runs/detect/yolo26s_det_grocery/results_summary.json
CSV row appended  -> runs/detect/all_runs.csv
Markdown report   -> /var/home/polygon/EE3703_RPC/runs/detect/yolo26s_det_grocery/results_summary.md

 All result files saved.


In [53]:
# Final recap printed inline 
print("\n" + "═" * 44)
print(f"  Run            : {RUN_NAME}")
print(f"  Completed at   : {summary['timestamp']}")
print(f"  Best weights   : {best_pt.name}")
print("  ─" * 22)
print(f"  mAP@0.50       : {summary['map50']}")
print(f"  mAP@0.50:0.95  : {summary['map50_95']}")
print(f"  Precision      : {summary['precision']}")
print(f"  Recall         : {summary['recall']}")
print("  ─" * 22)
for cls_name, ap in per_class_ap.items():
    print(f"  {cls_name:<22}: {ap}")
print("═" * 44)


════════════════════════════════════════════
  Run            : yolo26s_det_grocery
  Completed at   : 2026-04-02 22:34:12
  Best weights   : best.pt
  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─
  mAP@0.50       : 0.9661
  mAP@0.50:0.95  : 0.8505
  Precision      : 0.9461
  Recall         : 0.919
  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─
  Bottled_Drink         : 0.9734
  Canned                : 0.9676
  Fresh_Produce         : 0.9415
  Packaged_Food         : 0.9818
════════════════════════════════════════════
